In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import re
import os
import json

In [ ]:
game_date = '/home/abraham/uni/ikt453/project/v1/data/season'

from collections import defaultdict

def read_json(file_path: str):
    with open(file_path, 'r') as f:
        return json.load(f)

def game_date_iterator(directory: str):
    date_season_pattern = re.compile(r'games_(\d{4}-\d{2})_(\w+)')

    files = os.listdir(directory)
    files = list(map(lambda x: os.path.join(directory, x), files))

    for file in filter(lambda x: x.endswith('.json'), files):
        match = date_season_pattern.match(os.path.basename(file))
        season_label, season_type = match.groups()
        game_date_per_team = read_json(file)
        game_date_per_game_id = defaultdict(list)

        for gd in game_date_per_team:
            game_dt: str = gd['GAME_DATE']
            team_id: int = gd['TEAM_ID']
            team_av: str = gd['TEAM_ABBREVIATION']
            matchup: str = gd['MATCHUP']

            if 'vs.' in matchup:
                home_team, away_team = map(str.strip, matchup.split('vs.'))
            elif '@' in matchup:
                away_team, home_team = map(str.strip, matchup.split('@'))

            entry = (game_dt, home_team, away_team, season_label, season_type, (team_av, team_id))
            game_date_per_game_id[gd['GAME_ID']].append(entry)
        
        for game_id, entries in game_date_per_game_id.items():
            game_date, home_team, away_team, season_label, season_type, teams = map(set, zip(*entries))

            if len(game_date) != 1:
                raise ValueError(f"Inconsistent game dates for game_id {game_id}: {game_date}")
            if len(home_team) != 1:
                raise ValueError(f"Inconsistent home teams for game_id {game_id}: {home_team}")
            if len(away_team) != 1:
                raise ValueError(f"Inconsistent away teams for game_id {game_id}: {away_team}")
            if len(season_label) != 1:
                raise ValueError(f"Inconsistent seasons for game_id {game_id}: {season_label}")
            if len(season_type) != 1:
                raise ValueError(f"Inconsistent season types for game_id {game_id}: {season_type}")
            
            teams = dict(teams)

            home_team = home_team.pop()
            away_team = away_team.pop()

            yield dict(
                game_id=game_id,
                game_date=game_date.pop(),
                home_team=home_team,
                away_team=away_team,
                home_team_id=teams[home_team],
                away_team_id=teams[away_team],
                season_label=season_label.pop(),
                season_type=season_type.pop(),
                
            )

for row in game_date_iterator(game_date):
    # print(row)
    pass